### Tests the Marko parsing of footnotes
From [here](https://www.perplexity.ai/search/i-ve-attached-a-markdown-docum-5rfc3AkNSbOYE7_UeLn_GA) for example.

**NO AI MODEL CAN DO THIS...**

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict, Counter
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic
import re
from typing import Optional, Dict, List, Tuple
import datetime as dt

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz
import re

%load_ext autoreload
%autoreload 2

In [2]:
# # DOESN'T SUBS FOOTNOTES

# Markdown input
markdown_text = '''
<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" class="logo" width="120"/>

# Describe the current state of Seattle's Burke Gilman missing link bike trail, and explain how it reached its current state.

---
The "Missing Link" of Seattle's Burke-Gilman Trail remains incomplete as of February 2025. Despite being a priority in Seattle's Bicycle Master Plan since the 1990s, the Missing Link has faced resistance from local businesses along the proposed route and extensive litigation under the State Environmental Policy Act (SEPA), which opponents have used to delay progress[^1][^4][^9].

### Current State

- **Unfinished Section**: The Missing Link spans 1.4 miles around Salmon Bay, east of the Ballard Locks. Cyclists must navigate busy streets or alternative routes to bypass this gap[^1][^9].
- **Litigation and Delays**: Legal challenges have stalled construction repeatedly, making it one of Seattle's most prolonged infrastructure disputes[^1][^9].

### How It Reached This State

1. **Historical Background**: The Burke-Gilman Trail was initially developed on a former railroad corridor and opened in sections starting in 1978. While most of the 27-mile trail is complete, the Ballard segment has remained contentious due to its industrial setting[^5][^10].
3. **City Efforts**: Successive city administrations have allocated funds and conducted studies to close the gap. However, these efforts have been met with resistance, forcing SDOT to consider alternative routes[^7][^9].

The Missing Link remains a critical gap[^1][^7][^9].

<div style="text-align: center">⁂</div>

[^1]: https://www.youtube.com/watch?v=TKEdfRlFfnw
[^2]: https://www.theurbanist.org/2015/06/22/finding-the-burke-gilman-trails-missing-link/
[^3]: https://www.seattlebikeblog.com/2017/10/12/the-community-advised-missing-link-design-keeps-getting-better-for-everyone/
[^4]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/bgtmissinglink
[^5]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-multimodal-corridor/burke-gilman-trail-history
[^6]: https://www.traillink.com/trail/burke-gilman-trail/
[^7]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-bike-route-study
[^8]: https://www.seattlebikeblog.com/2015/03/31/cascade-publishes-very-very-long-timeline-of-work-to-build-the-burke-gilman-missing-link/
[^9]: https://www.seattletimes.com/seattle-news/transportation/is-the-burke-gilman-missing-link-in-seattle-finally-getting-built/
[^10]: https://www.seattlepi.com/seattlenews/article/History-Burke-Gilman-bicycle-walk-trail-Seattle-14369100.php
'''

# import marko

# import marko

# import marko

# def parse_markdown_perplex_content(markdown_text):
#     # Initialize Marko with footnote extension
#     markdown = marko.Markdown(extensions=['footnote'])

#     # Parse the markdown
#     parsed_doc = markdown.parse(markdown_text)

#     # 1. Get first level 1 headline
#     headline = None
#     for element in parsed_doc.children:
#         if isinstance(element, marko.block.Heading) and element.level == 1:
#             headline = ''.join(str(child.children) for child in element.children)
#             break

#     # 2. Create footnote dictionary by parsing the footnote definitions directly
#     footnote_dict = {}
#     import re

#     # Use regex to find footnote definitions
#     footnote_pattern = r'\[\^(\d+)\]:\s*(http[s]?://\S+)'
#     for match in re.finditer(footnote_pattern, markdown_text):
#         number, url = match.groups()
#         footnote_dict[number] = url

#     # 3. Replace footnotes with markdown links in the entire document (rendered_text)
#     rendered = markdown_text
#     for number, url in footnote_dict.items():
#         rendered = rendered.replace(f'[^{number}]', f'[{number}]({url})')

#     # 4. Extract full_prompt (text of heading 1 + text below it until last "---")
#     full_prompt = None
#     ai_response = None
#     rendered_ai_response = None

#     if headline:
#         # Find the start of the first-level heading and extract text below it until "---"
#         heading_start_index = markdown_text.find(f"# {headline}")
#         divider_index = markdown_text.rfind('---')  # Find last occurrence of "---"

#         if heading_start_index != -1 and divider_index != -1:
#             # Extract text from just below the heading to the last divider
#             full_prompt_start_index = heading_start_index + len(f"# {headline}")
#             full_prompt = f"{headline}\n" + markdown_text[full_prompt_start_index:divider_index].strip()

#             # Extract ai_response (text from last divider "---" to footnotes)
#             ai_response_start_index = divider_index + len('---')
#             footnotes_start_index = markdown_text.find('[^')  # Footnotes typically start with "[^"
            
#             if footnotes_start_index == -1:  # If no footnotes exist, take all remaining text
#                 ai_response = markdown_text[ai_response_start_index:].strip()
#             else:
#                 ai_response = markdown_text[ai_response_start_index:footnotes_start_index].strip()

#             # Render ai_response (replace footnotes with links)
#             rendered_ai_response = ai_response
#             for number, url in footnote_dict.items():
#                 rendered_ai_response = rendered_ai_response.replace(f'[^{number}]', f'[{number}]({url})')

#     result = {
#         'headline': headline,
#         'footnotes': footnote_dict,
#         'rendered_text': rendered,
#         'full_prompt': full_prompt,
#         'ai_response': ai_response,
#         'rendered_ai_response': rendered_ai_response
#     }
    
#     return result

# def parse_markdown_perplex_file(file_path):

#     markdown_text = lpz.read_markdown_file(file_path)
    
#     return parse_markdown_perplex_content(markdown_text)


In [9]:
from markdown_it import MarkdownIt
from mdit_py_plugins.footnote import footnote_plugin
import re

def process_markdown(md_text):
    # Initialize parser with footnote plugin
    md = MarkdownIt().use(footnote_plugin)
    env = {}
    tokens = md.parse(md_text, env)
    
    results = {
        'first_h1': None,
        'footnotes': {},
        'modified_text': md_text
    }
    
    # 1. Find first H1 heading
    for i, token in enumerate(tokens):
        if token.type == 'heading_open' and token.tag == 'h1':
            if tokens[i+1].type == 'inline':
                results['first_h1'] = tokens[i+1].content
                break
    ic(env['footnotes']['list'])
    # 2. Extract footnotes from env
    if 'footnotes' in env and 'list' in env['footnotes']:
        for i, footnote in enumerate(env['footnotes']['list']):
            # Check if footnote has content
            if footnote['content']:
                # Extract the URL from the tokens within the footnote content
                url = footnote['content']
                results['footnotes'][str(i + 1)] = url  # Footnote keys start from 1

    # 3. Replace footnote references with markdown links in the main text
    def replace_footnotes(match):
        fn_id = match.group(1)
        if fn_id in results['footnotes']:
            return f"[{fn_id}]({results['footnotes'][fn_id]})"
        return match.group(0)

    results['modified_text'] = re.sub(r'\[\^(\d+)\]', replace_footnotes, results['modified_text'])

    # 4. Replace footnote definitions with markdown links at the bottom
    def replace_footnote_definitions(match):
        fn_id = match.group(1)
        if fn_id in results['footnotes']:
            url = results['footnotes'][fn_id]
            return f"[^{fn_id}]: [{fn_id}]({url}): {url}"
        return match.group(0)

    results['modified_text'] = re.sub(r'\[\^(\d+)\]:\s*(.*)', replace_footnote_definitions, results['modified_text'], flags=re.MULTILINE)
    
    return results


# Example usage:
markdown_text = """
<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" class="logo" width="120"/>

# Describe the current state of Seattle's Burke Gilman missing link bike trail, and explain how it reached its current state.

---
The "Missing Link" of Seattle's Burke-Gilman Trail remains incomplete as of February 2025. Despite being a priority in Seattle's Bicycle Master Plan since the 1990s, the Missing Link has faced resistance from local businesses along the proposed route and extensive litigation under the State Environmental Policy Act (SEPA), which opponents have used to delay progress[^1][^4][^9].

### Current State

- **Unfinished Section**: The Missing Link spans 1.4 miles around Salmon Bay, east of the Ballard Locks. Cyclists must navigate busy streets or alternative routes to bypass this gap[^1][^9].
- **Litigation and Delays**: Legal challenges have stalled construction repeatedly, making it one of Seattle's most prolonged infrastructure disputes[^1][^9].

### How It Reached This State

1. **Historical Background**: The Burke-Gilman Trail was initially developed on a former railroad corridor and opened in sections starting in 1978. While most of the 27-mile trail is complete, the Ballard segment has remained contentious due to its industrial setting[^5][^10].
3. **City Efforts**: Successive city administrations have allocated funds and conducted studies to close the gap. However, these efforts have been met with resistance, forcing SDOT to consider alternative routes[^7][^9].

The Missing Link remains a critical gap[^1][^7][^9].

<div style="text-align: center">⁂</div>

[^1]: https://www.youtube.com/watch?v=TKEdfRlFfnw

[^2]: https://www.theurbanist.org/2015/06/22/finding-the-burke-gilman-trails-missing-link/

[^3]: https://www.seattlebikeblog.com/2017/10/12/the-community-advised-missing-link-design-keeps-getting-better-for-everyone/

[^4]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/bgtmissinglink

[^5]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-multimodal-corridor/burke-gilman-trail-history

[^6]: https://www.traillink.com/trail/burke-gilman-trail/

[^7]: https://www.seattle.gov/transportation/projects-and-programs/programs/bike-program/ballard-bike-route-study

[^8]: https://www.seattlebikeblog.com/2015/03/31/cascade-publishes-very-very-long-timeline-of-work-to-build-the-burke-gilman-missing-link/

[^9]: https://www.seattletimes.com/seattle-news/transportation/is-the-burke-gilman-missing-link-in-seattle-finally-getting-built/

[^10]: https://www.seattlepi.com/seattlenews/article/History-Burke-Gilman-bicycle-walk-trail-Seattle-14369100.php
"""

output = process_markdown(markdown_text)

print("1. First H1:", output['first_h1'])
print("\n2. Footnotes:", output['footnotes'])
print("\n3. Modified Text:\n", output['modified_text'])


ic| env['footnotes']['list']: {0: {'count': 4, 'label': '

1'},
                               1: {'count': 1, 'label': '4'},
                               2: {'count': 5, 'label': '9'},
                               3: {'count': 1, 'label': '5'},
                               4: {'count': 1, 'label': '10'},
                               5: {'count': 2, 'label': '7'}}


TypeError: 'int' object is not subscriptable

In [4]:
datdir = rfw.refwrangle_test_dir / 'dat'

# file_path_perp = datdir / "perplexity_example.md"
# file_path_smc = datdir / "perplexity_single_prompt_savemychatbot_example.md"
# file_path_not_md = datdir / 'obsnotecitekeys.csv'
# file_path_no_exist = datdir / "__alsdkfjasd__.md"
tmpdir = pl.Path(r'~/tmp')

file_path = pl.Path(r"C:\Users\scott\tmp\describe_perplex.md")
file_path = pl.Path(r"C:\Users\scott\tmp\Are any bears there_.md")

markdown_text = lpz.read_markdown_file(file_path)
result = process_markdown(markdown_text)

In [5]:
result

{'first_h1': 'Are any bears there?',
 'footnotes': {},
 'modified_text': '<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" class="logo" width="120"/>\n\n# Are any bears there?\n\n---\nYes, there are bears in Olympic National Park. Specifically, the park is home to **black bears**, which are common throughout the region. These bears are smaller and darker than their grizzly or brown bear relatives, and they are the only bear species found in the park. Grizzly bears are not present in Olympic National Park[^1][^2][^3].\n\nBlack bears can be spotted in various areas of the park, including Hurricane Ridge, the Hoh Rainforest, Enchanted Valley, and the Sol Duc Valley. They are most active during dawn and dusk, and sightings are more likely in spring and fall when they forage for food like berries, fish, and insects[^1][^4].\n\nWhile black bears generally avoid human interaction, it’s important to exercise caution. Visitors should store food securely, avoid approac

In [6]:
#result

In [13]:
from marko import Markdown
from marko.ext.footnote import make_extension

# Input Markdown document with footnotes
input_markdown = """
This is an example of a document with footnotes[^1]. You can use footnotes to provide additional context or references[^2].

[^1]: https://example.com/footnote1
[^2]: https://example.com/footnote2
"""

def convert_footnotes_to_links(markdown_text):
    # Initialize Marko with the footnote extension
    markdown = Markdown(extensions=['footnote'])
    
    # Parse the input Markdown into an AST
    doc = markdown.parse(markdown_text)
    
    # Extract footnotes and their content
    footnotes = {}
    for child in doc.children:
        if hasattr(child, 'element') and child.element == 'footnote_definition':
            # Get the footnote label and content
            label = child.label
            # Extract the text content from the first paragraph of the footnote
            content = child.children[0].children[0].children
            footnotes[label] = content
    
    # Create a new renderer that converts footnote references to links
    class LinkRenderer(markdown.renderer):
        def render_footnote_reference(self, element):
            if element.label in footnotes:
                return f'[{element.label}]({footnotes[element.label]})'
            return super().render_footnote_reference(element)
    
    # Render with our custom renderer
    markdown.renderer = LinkRenderer
    main_text = markdown.render(doc)
    
    # Generate the footnote list
    footnote_list = "\n".join(
        f"[{label}]: {content}" for label, content in footnotes.items()
    )
    
    return f"{main_text}\n\n---\n\n### Footnotes\n\n{footnote_list}"

# Convert and print the result
output = convert_footnotes_to_links(input_markdown)
print(output)


TypeError: FootnoteRendererMixin.__init__() takes 1 positional argument but 4 were given